In [1]:
import numpy as np
import pandas as pd

In [4]:
np.random.seed(23)
mu_vec1 = np.array([0, 0, 0])
cov_mat1 = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
class1_sample = np.random.multivariate_normal(mu_vec1, cov_mat1, 20)

df = pd.DataFrame(class1_sample, columns=['X1', 'X2', 'X3'])
df['target'] = 1

mu_vec2 = np.array([1, 1, 1])
cov_mat2 = np.array([[1, 0, 0], [0, 1, 0], [0, 0, 1]])
class2_sample = np.random.multivariate_normal(mu_vec2, cov_mat2, 20)

df2 = pd.DataFrame(class2_sample, columns=['X1', 'X2', 'X3'])
df2['target'] = 0

df = pd.concat([df, df2], ignore_index=True)
df = df.sample(40)

In [5]:
df.head()

,X1,X2,X3,target
2,-0.367548,-1.137460,-1.322148,1
34,0.177061,-0.598109,1.226512,0
14,0.420623,0.411620,-0.071324,1
11,1.968435,-0.547788,-0.679418,1
12,-2.506230,0.146960,0.606195,1


In [7]:
%pip install plotly
import plotly.express as px
fig = px.scatter_3d(df, x='X1', y='X2', z='X3', color='target')
fig.show()

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 831.8 kB/s eta 0:00:0000:0100:01

[notice] A new release of pip is available: 24.2 -> 26.1.1
[notice] To update, run: pip install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [8]:
# step 1 apply standard scaling
from sklearn.preprocessing import StandardScaler
scaler = StandardScaler()
X_scaled = scaler.fit_transform(df[['X1', 'X2', 'X3']])


In [9]:
#step 2 create a covariance matrix
cov_mat = np.cov(X_scaled.T)
print('Covariance matrix \n%s' % cov_mat)

Covariance matrix 
[[1.02564103 0.20478114 0.080118  ]
 [0.20478114 1.02564103 0.19838882]
 [0.080118   0.19838882 1.02564103]]


In [10]:
eigen_values, eigen_vectors = np.linalg.eig(cov_mat)
print('Eigenvectors \n%s' % eigen_vectors)
print('\nEigenvalues \n%s' % eigen_values)

Eigenvectors 
[[-0.53875915 -0.69363291  0.47813384]
 [-0.65608325 -0.01057596 -0.75461442]
 [-0.52848211  0.72025103  0.44938304]]

Eigenvalues 
[1.3536065  0.94557084 0.77774573]


In [14]:
import plotly.graph_objects as go

# Extract the 3 eigenvectors (columns of the eigenvectors matrix)
ev1 = eigen_vectors[:, 0]
ev2 = eigen_vectors[:, 1]
ev3 = eigen_vectors[:, 2]

# Create 3D plot
fig = go.Figure()

# Plot data points for class 0
class_0_indices = df['target'] == 0
fig.add_trace(go.Scatter3d(
    x=X_scaled[class_0_indices, 0],
    y=X_scaled[class_0_indices, 1],
    z=X_scaled[class_0_indices, 2],
    mode='markers',
    name='Class 0',
    marker=dict(size=5, color='lightblue', opacity=0.6)
))

# Plot data points for class 1
class_1_indices = df['target'] == 1
fig.add_trace(go.Scatter3d(
    x=X_scaled[class_1_indices, 0],
    y=X_scaled[class_1_indices, 1],
    z=X_scaled[class_1_indices, 2],
    mode='markers',
    name='Class 1',
    marker=dict(size=5, color='orange', opacity=0.6)
))

# Plot eigenvector 1
fig.add_trace(go.Scatter3d(
    x=[0, ev1[0]],
    y=[0, ev1[1]],
    z=[0, ev1[2]],
    mode='lines+markers',
    name='EV1 (PC1)',
    line=dict(color='red', width=5),
    marker=dict(size=8, color='red')
))

# Plot eigenvector 2
fig.add_trace(go.Scatter3d(
    x=[0, ev2[0]],
    y=[0, ev2[1]],
    z=[0, ev2[2]],
    mode='lines+markers',
    name='EV2 (PC2)',
    line=dict(color='green', width=5),
    marker=dict(size=8, color='green')
))

# Plot eigenvector 3
fig.add_trace(go.Scatter3d(
    x=[0, ev3[0]],
    y=[0, ev3[1]],
    z=[0, ev3[2]],
    mode='lines+markers',
    name='EV3 (PC3)',
    line=dict(color='blue', width=5),
    marker=dict(size=8, color='blue')
))

fig.update_layout(
    title='Data Points with Eigenvectors (Principal Components)',
    scene=dict(
        xaxis_title='X1',
        yaxis_title='X2',
        zaxis_title='X3',
        camera=dict(eye=dict(x=1.5, y=1.5, z=1.5))
    ),
    height=700,
    showlegend=True
)

fig.show()

In [21]:
eigen_vectors.shape
pc = eigen_vectors[0:2]
pc.shape

(2, 3)

In [22]:
transformed_df = np.dot(X_scaled, pc.T)
transformed = pd.DataFrame(transformed_df, columns=['PC1', 'PC2'])
transformed['target'] = df['target']
transformed.head()

,PC1,PC2,target
0,0.599433,1.795862,1
1,1.056919,-0.212737,1
2,-0.271876,0.498222,1
3,-0.621586,0.023110,1
4,1.567286,1.730967,1


In [25]:
fig = px.scatter(transformed, x='PC1', y='PC2', color='target')
fig.show()